In [ ]:
# ============================================================
# CÉLULA 1 — CARREGAMENTO DO AGENTE PLANAPP
# ============================================================
#%pip install "mcp"
import importlib
import agent_jupyter

agent_jupyter = importlib.reload(agent_jupyter)

PlanAppAgent = agent_jupyter.PlanAppAgent

In [ ]:
# ============================================================
# PLANAPP AI — JUPYTER
# CÉLULA 3 — INTERFACE DO AGENTE
# ============================================================

import asyncio
import html

import ipywidgets as widgets
from IPython.display import display


# ============================================================
# TÍTULO
# ============================================================

titulo = widgets.HTML(
    value="""
    <h2 style="margin-bottom:5px;">
        🤖 PlanApp AI
    </h2>
    <div style="color:#666;">
        Agente de planejamento e análise de enlaces
    </div>
    """
)


# ============================================================
# ENTRADA
# ============================================================

entrada = widgets.Textarea(
    value="",
    placeholder=(
        "Exemplo:\n"
        "Analise um enlace entre a Praça da Sé "
        "e o Largo do Paissandu em São Paulo."
    ),
    layout=widgets.Layout(
        width="100%",
        height="90px"
    )
)


# ============================================================
# BOTÕES
# ============================================================

button = widgets.Button(
    description="🚀 Analisar enlace",
    button_style="primary",
    layout=widgets.Layout(
        width="180px"
    )
)

button_nova = widgets.Button(
    description="🔄 Nova análise",
    layout=widgets.Layout(
        width="140px"
    )
)


# ============================================================
# STATUS ATUAL
# ============================================================

status = widgets.HTML(
    value="""
    <div style="
        padding:10px;
        margin-top:10px;
        border-radius:6px;
        background:#f5f5f5;
    ">
        Aguardando análise...
    </div>
    """
)


# ============================================================
# HISTÓRICO VISÍVEL DA EXECUÇÃO
# ============================================================

historico_status = []


log_execucao = widgets.HTML(
    value="",
    layout=widgets.Layout(
        width="100%"
    )
)


# ============================================================
# RESULTADO FINAL
# ============================================================

resposta = widgets.HTML(
    value="",
    layout=widgets.Layout(
        width="100%"
    )
)


# ============================================================
# ESTADO
# ============================================================

executando = False
task_atual = None


# ============================================================
# ATUALIZA STATUS
# ============================================================

def atualizar_status(
    texto,
    tipo="processing"
):

    global historico_status

    if texto is None:
        return

    texto = str(texto)

    # --------------------------------------------------------
    # Evita duplicar exatamente a mesma mensagem consecutiva
    # --------------------------------------------------------

    if (
        historico_status
        and historico_status[-1] == texto
    ):
        return

    historico_status.append(
        texto
    )

    # --------------------------------------------------------
    # Status atual
    # --------------------------------------------------------

    texto_html = html.escape(
        texto
    )

    if tipo == "success":

        fundo = "#e8f5e9"

    elif tipo == "error":

        fundo = "#ffebee"

    else:

        fundo = "#f5f5f5"

    status.value = f"""
    <div style="
        padding:10px;
        margin-top:10px;
        border-radius:6px;
        background:{fundo};
        font-weight:500;
    ">
        {texto_html}
    </div>
    """

    # --------------------------------------------------------
    # Histórico da execução
    # --------------------------------------------------------

    linhas = []

    for item in historico_status:

        item_html = html.escape(
            item
        )

        # Espaçamento antes de cada etapa
        if item.startswith("🔵 Etapa"):

            linhas.append(
                f"""
                <div style="
                    margin-top:10px;
                    margin-bottom:4px;
                    font-weight:600;
                ">
                    {item_html}
                </div>
                """
            )

        elif item.startswith("🔧"):

            linhas.append(
                f"""
                <div style="
                    margin-left:14px;
                    margin-top:5px;
                    font-family:monospace;
                ">
                    {item_html}
                </div>
                """
            )

        elif item.startswith("📥"):

            linhas.append(
                f"""
                <div style="
                    margin-left:32px;
                    color:#555;
                    font-family:monospace;
                    white-space:pre-wrap;
                ">
                    {item_html}
                </div>
                """
            )

        else:

            linhas.append(
                f"""
                <div style="
                    margin-top:5px;
                ">
                    {item_html}
                </div>
                """
            )

    log_execucao.value = f"""
    <div style="
        margin-top:12px;
        padding:12px;
        border:1px solid #ddd;
        border-radius:6px;
        background:#fafafa;
    ">
        <div style="
            font-weight:600;
            margin-bottom:8px;
        ">
            🔄 Execução do agente
        </div>

        {''.join(linhas)}
    </div>
    """


# ============================================================
# AGENTE
# ============================================================

agent = PlanAppAgent(
    progress_callback=atualizar_status
)


# ============================================================
# EXECUÇÃO ASSÍNCRONA
# ============================================================

async def executar_analise_async(
    texto
):

    global executando

    try:

        # ----------------------------------------------------
        # NÃO limpar aqui.
        #
        # O primeiro "Iniciando agente Qwen..."
        # já foi colocado pelo botão.
        # ----------------------------------------------------

        resultado = await agent.ask(
            texto
        )

        # ----------------------------------------------------
        # Resultado final
        # ----------------------------------------------------

        resposta.value = f"""
        <div style="
            margin-top:18px;
            padding:15px;
            border:1px solid #ddd;
            border-radius:6px;
            background:white;
        ">

            <h3 style="
                margin-top:0;
                margin-bottom:12px;
            ">
                📊 Resultado da análise
            </h3>

            <div style="
                white-space:pre-wrap;
                line-height:1.5;
            ">
                {html.escape(str(resultado))}
            </div>

        </div>
        """

    except Exception as e:

        resposta.value = f"""
        <div style="
            margin-top:18px;
            padding:15px;
            border:1px solid #f5c2c7;
            border-radius:6px;
            background:#fff5f5;
        ">

            <h3 style="
                margin-top:0;
            ">
                ❌ Erro na análise
            </h3>

            <div style="
                white-space:pre-wrap;
            ">
                {html.escape(str(e))}
            </div>

        </div>
        """

    finally:

        executando = False

        button.disabled = False

        button_nova.disabled = False


# ============================================================
# BOTÃO ANALISAR
# ============================================================

def executar_analise(
    b
):

    global executando
    global task_atual
    global historico_status

    if executando:
        return

    texto = entrada.value.strip()

    if not texto:

        atualizar_status(
            "⚠️ Digite uma solicitação para iniciar a análise.",
            "error"
        )

        return

    executando = True

    button.disabled = True
    button_nova.disabled = True

    # --------------------------------------------------------
    # Limpa execução anterior
    # --------------------------------------------------------

    historico_status.clear()

    log_execucao.value = ""

    resposta.value = ""

    # --------------------------------------------------------
    # Primeira mensagem
    # --------------------------------------------------------

    atualizar_status(
        "🔵 Iniciando agente Qwen...",
        "processing"
    )

    # --------------------------------------------------------
    # Cria task assíncrona
    # --------------------------------------------------------

    task_atual = asyncio.create_task(
        executar_analise_async(
            texto
        )
    )


# ============================================================
# NOVA ANÁLISE
# ============================================================

def nova_analise(
    b
):

    global historico_status

    if executando:
        return

    entrada.value = ""

    resposta.value = ""

    historico_status.clear()

    log_execucao.value = ""

    status.value = """
    <div style="
        padding:10px;
        margin-top:10px;
        border-radius:6px;
        background:#f5f5f5;
    ">
        Aguardando análise...
    </div>
    """


# ============================================================
# CALLBACKS
# ============================================================

button.on_click(
    executar_analise
)

button_nova.on_click(
    nova_analise
)


# ============================================================
# EXEMPLOS
# ============================================================

exemplos = widgets.HTML(
    value="""
    <div style="
        margin-top:12px;
        color:#666;
        font-size:13px;
    ">

        <b>Exemplos:</b>

        <ul>
            <li>
                Analise um enlace entre a Praça da Sé
                e o Largo do Paissandu em São Paulo.
            </li>

            <li>
                Analise um enlace entre Curitiba
                e São José dos Pinhais.
            </li>

            <li>
                Verifique a viabilidade de um enlace
                entre dois pontos informados.
            </li>
        </ul>

    </div>
    """
)


# ============================================================
# LAYOUT
# ============================================================

botoes = widgets.HBox(
    [
        button,
        button_nova
    ],
    layout=widgets.Layout(
        margin="10px 0"
    )
)


display(
    titulo,
    entrada,
    botoes,
    status,
    log_execucao,
    resposta,
    exemplos
)